[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/03_attention_variants/03.3_gqa_deep_dive/lab.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/03_attention_variants/03.3_gqa_deep_dive/lab.ipynb)

# 3.3 Lab: GQA Deep Dive — Empirical Verification


**Goal:** Empirically verify GQA theory from Module 3.2 using Mistral-7B weights.
We will: (1) prove GQA from weight shapes, (2) measure real KV cache memory,
(3) compare measured vs theoretical, (4) demonstrate tensor parallelism sharding.


In [ ]:
# Setup: clone repo utilities and install deps
import subprocess, sys, os
if not os.path.exists("/tmp/llm-inference-at-scale"):
    subprocess.run(["git", "clone", "--depth=1",
                    "https://github.com/harshuljain13/llm-inference-at-scale.git",
                    "/tmp/llm-inference-at-scale"], check=True)
sys.path.insert(0, "/tmp/llm-inference-at-scale")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torch", "transformers", "matplotlib", "numpy"], check=True)

import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoConfig, AutoModelForCausalLM
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")


## 1. Proving Mistral-7B Uses GQA from Weight Shapes

If Q heads ≠ KV heads, the model uses GQA. We inspect the actual parameter tensors.


In [ ]:
# Load Mistral-7B config and prove GQA from architecture
config = AutoConfig.from_pretrained("mistralai/Mistral-7B-v0.1")

# Extract attention geometry
n_q_heads = config.num_attention_heads       # Query heads
n_kv_heads = config.num_key_value_heads      # KV heads (fewer = GQA)
head_dim = config.hidden_size // n_q_heads   # Dimension per head
n_layers = config.num_hidden_layers

print(f"Model: Mistral-7B-v0.1")
print(f"Hidden size: {config.hidden_size}")
print(f"Query heads: {n_q_heads}")
print(f"KV heads: {n_kv_heads}")
print(f"Head dim: {head_dim}")
print(f"Layers: {n_layers}")
print(f"\nGQA ratio: {n_q_heads // n_kv_heads} query heads per KV head")
print(f"\n{'✅ CONFIRMED GQA' if n_kv_heads < n_q_heads else '❌ Not GQA'}: "
      f"{n_kv_heads} KV heads < {n_q_heads} Q heads")


In [ ]:
# Load model weights and verify projection dimensions match GQA
model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-v0.1",
    dtype=torch.float16,
    device_map="auto"
)

# Inspect layer 0 attention projections
attn = model.model.layers[0].self_attn
q_shape = attn.q_proj.weight.shape  # (n_q_heads * head_dim, hidden)
k_shape = attn.k_proj.weight.shape  # (n_kv_heads * head_dim, hidden)
v_shape = attn.v_proj.weight.shape  # (n_kv_heads * head_dim, hidden)

print(f"Q projection: {q_shape} -> {q_shape[0]//head_dim} heads x {head_dim}d")
print(f"K projection: {k_shape} -> {k_shape[0]//head_dim} heads x {head_dim}d")
print(f"V projection: {v_shape} -> {v_shape[0]//head_dim} heads x {head_dim}d")
print(f"\n✅ K,V have {k_shape[0]//head_dim} heads (not {q_shape[0]//head_dim})")
print(f"   Memory saved: {(1 - n_kv_heads/n_q_heads)*100:.0f}% less KV cache vs MHA")


## 2. Measuring Real KV Cache Memory at Different Sequence Lengths

We run actual forward passes and measure GPU memory consumed by the KV cache.


In [ ]:
# Measure KV cache memory at increasing sequence lengths
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1")
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Baseline memory (model weights only)
baseline_mem = torch.cuda.memory_allocated() / 1024**2  # MB

seq_lengths = [128, 256, 512, 1024, 2048]
measured_kv_mb = []

for seq_len in seq_lengths:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    before = torch.cuda.memory_allocated()

    # Generate tokens to build KV cache
    input_ids = torch.randint(1, 30000, (1, seq_len), device="cuda")
    with torch.no_grad():
        outputs = model(input_ids, use_cache=True)
        # KV cache is stored in outputs.past_key_values
        past = outputs.past_key_values

    after = torch.cuda.memory_allocated()
    kv_mem = (after - before) / 1024**2  # MB (includes activations overhead)
    measured_kv_mb.append(kv_mem)
    print(f"Seq {seq_len:5d}: KV cache + activations = {kv_mem:.1f} MB")

    del outputs, past, input_ids
    torch.cuda.empty_cache()


## 3. Theoretical vs Measured KV Cache

From Module 3.2, the theoretical KV cache formula:
```
KV_bytes = 2 × n_kv_heads × head_dim × n_layers × seq_len × bytes_per_param
```
Factor of 2 accounts for both K and V tensors.


In [ ]:
# Theoretical KV cache size (from 03.2 formula)
bytes_per_param = 2  # FP16

theoretical_kv_mb = []
for seq_len in seq_lengths:
    # 2 (K+V) * n_kv_heads * head_dim * n_layers * seq_len * dtype_bytes
    kv_bytes = 2 * n_kv_heads * head_dim * n_layers * seq_len * bytes_per_param
    theoretical_kv_mb.append(kv_bytes / 1024**2)

# Per-token KV cache
per_token_bytes = 2 * n_kv_heads * head_dim * n_layers * bytes_per_param
print(f"Per-token KV cache: {per_token_bytes:,} bytes = {per_token_bytes/1024:.1f} KB")
print(f"  (8 KV heads × 128d × 32 layers × 2 bytes × 2 tensors)\n")

print(f"{'Seq Len':>8} {'Theoretical':>12} {'Measured':>10} {'Overhead':>10}")
print("-" * 44)
for i, sl in enumerate(seq_lengths):
    overhead = measured_kv_mb[i] - theoretical_kv_mb[i]
    print(f"{sl:>8} {theoretical_kv_mb[i]:>10.1f} MB {measured_kv_mb[i]:>8.1f} MB {overhead:>8.1f} MB")


In [ ]:
# Plot: Theoretical vs Measured KV cache scaling
fig_5, ax_5 = plt.subplots(1, 1, figsize=(8, 5))

ax_5.plot(seq_lengths, theoretical_kv_mb, 'b-o', label='Theoretical (formula)', linewidth=2)
ax_5.plot(seq_lengths, measured_kv_mb, 'r--s', label='Measured (GPU)', linewidth=2)

ax_5.set_xlabel("Sequence Length (tokens)", fontsize=12)
ax_5.set_ylabel("Memory (MB)", fontsize=12)
ax_5.set_title("GQA KV Cache: Theoretical vs Measured (Mistral-7B, FP16)", fontsize=13)
ax_5.legend(fontsize=11)
ax_5.grid(True, alpha=0.3)

# Annotate per-token cost
ax_5.annotate(f"{per_token_bytes/1024:.0f} KB/token",
            xy=(seq_lengths[-1], theoretical_kv_mb[-1]),
            xytext=(-80, 20), textcoords='offset points',
            fontsize=10, arrowprops=dict(arrowstyle='->', color='blue'))
plt.tight_layout()
plt.savefig("kv_cache_scaling.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: kv_cache_scaling.png")


## 3b. GQA vs MHA Memory Savings

What if Mistral-7B used full MHA (32 KV heads) instead of GQA (8 KV heads)?


In [ ]:
# Compare GQA (8 heads) vs hypothetical MHA (32 heads)
mha_kv_mb = [t * (n_q_heads / n_kv_heads) for t in theoretical_kv_mb]  # 4x more

fig_6, ax_6 = plt.subplots(figsize=(8, 5))
ax_6.plot(seq_lengths, mha_kv_mb, 'r-^', label=f'MHA ({n_q_heads} KV heads)', linewidth=2)
ax_6.plot(seq_lengths, theoretical_kv_mb, 'b-o', label=f'GQA ({n_kv_heads} KV heads)', linewidth=2)
ax_6.fill_between(seq_lengths, theoretical_kv_mb, mha_kv_mb, alpha=0.15, color='green')

# Annotate savings
mid = len(seq_lengths) // 2
savings = mha_kv_mb[mid] - theoretical_kv_mb[mid]
ax_6.annotate(f"{savings:.0f} MB saved\n({(1-n_kv_heads/n_q_heads)*100:.0f}% reduction)",
            xy=(seq_lengths[mid], (mha_kv_mb[mid]+theoretical_kv_mb[mid])/2),
            fontsize=11, ha='center', color='green', fontweight='bold')

ax_6.set_xlabel("Sequence Length (tokens)", fontsize=12)
ax_6.set_ylabel("KV Cache Memory (MB)", fontsize=12)
ax_6.set_title("GQA Memory Savings vs MHA (Mistral-7B, FP16)", fontsize=13)
ax_6.legend(fontsize=11)
ax_6.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("gqa_vs_mha.png", dpi=150, bbox_inches='tight')
plt.show()


## 4. Tensor Parallelism: Why 8 KV Heads Divides Cleanly

GQA's 8 KV heads enable clean sharding across 1, 2, 4, or 8 GPUs
with zero padding or replication overhead.


In [ ]:
# Tensor parallelism sharding analysis
gpu_counts = [1, 2, 4, 8]

print(f"Mistral-7B GQA: {n_kv_heads} KV heads, {n_q_heads} Q heads\n")
print(f"{'GPUs':>4} {'KV heads/GPU':>13} {'Q heads/GPU':>12} {'KV MB/GPU (2K)':>15} {'Clean?':>7}")
print("-" * 55)

seq_2k_kv_mb = 2 * n_kv_heads * head_dim * n_layers * 2048 * bytes_per_param / 1024**2

for gpus in gpu_counts:
    kv_per_gpu = n_kv_heads // gpus
    q_per_gpu = n_q_heads // gpus
    mem_per_gpu = seq_2k_kv_mb / gpus
    clean = "✅" if n_kv_heads % gpus == 0 else "❌"
    print(f"{gpus:>4} {kv_per_gpu:>13} {q_per_gpu:>12} {mem_per_gpu:>13.1f} MB {clean:>7}")

# Also check problematic GPU counts
print(f"\n--- Non-power-of-2 GPUs (uneven sharding) ---")
for gpus in [3, 5, 6, 7]:
    remainder = n_kv_heads % gpus
    status = "✅ even" if remainder == 0 else f"❌ remainder={remainder}"
    print(f"{gpus} GPUs: 8 KV heads ÷ {gpus} = {status}")


In [ ]:
# Visualize tensor parallelism sharding
fig_8, axes = plt.subplots(1, 4, figsize=(14, 3))

for idx, gpus in enumerate(gpu_counts):
    ax_8 = axes[idx]
    kv_per_gpu = n_kv_heads // gpus
    # Draw KV head blocks colored by GPU assignment
    colors = plt.cm.Set3(np.linspace(0, 1, 8))
    for head_idx in range(n_kv_heads):
        gpu_id = head_idx // kv_per_gpu
        ax_8.barh(0, 1, left=head_idx, color=colors[gpu_id], edgecolor='black', linewidth=0.8)
        ax_8.text(head_idx + 0.5, 0, str(head_idx), ha='center', va='center', fontsize=8)

    ax_8.set_xlim(0, 8)
    ax_8.set_ylim(-0.5, 0.5)
    ax_8.set_title(f"{gpus} GPU{'s' if gpus>1 else ''}: {kv_per_gpu} heads each", fontsize=10)
    ax_8.set_xlabel("KV Head Index")
    ax_8.set_yticks([])

plt.suptitle("KV Head Sharding Across GPUs (Mistral-7B GQA)", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("tp_sharding.png", dpi=150, bbox_inches='tight')
plt.show()


## Key Findings

| Metric | Value |
|--------|-------|
| KV heads (GQA) | 8 (vs 32 query heads) |
| Per-token KV cache | 131 KB (FP16) |
| Memory savings vs MHA | 75% |
| Clean TP sharding | 1, 2, 4, 8 GPUs |
| KV cache at 2K tokens | ~8 MB per sequence |

**Takeaway:** GQA is not just a memory optimization. The 8-head design
is an engineering choice that enables clean tensor parallelism while
reducing KV cache by 75% with minimal quality loss.


In [ ]:
# Cleanup GPU memory
del model
torch.cuda.empty_cache()
print("Done. GPU memory released.")
